# Probability Concepts & Bayes' Theorem (CFA Level 1)
## Foundations of Probability for Portfolio Management and Risk Analysis

---

### Why Probability Matters in Finance

Every investment decision is a bet on the future -- and the future is uncertain. Probability is the mathematical language we use to reason about that uncertainty.

When a portfolio manager says "there's a 20% chance of recession," or a credit analyst estimates "the probability of default is 2%," they are using probability concepts. Without this framework, investment decisions would be pure guesswork.

> **Key Concept:** Probability quantifies uncertainty. In finance, it is the foundation for risk assessment, portfolio construction, option pricing, and credit analysis. Every "what could go wrong?" question is fundamentally a probability question.

**What you will learn:**
1. Probability axioms, addition and multiplication rules
2. Conditional probability and independence
3. Bayes' theorem: derivation, implementation, and financial applications
4. Expected value, variance, covariance, and portfolio theory foundations
5. Counting methods for combinatorial finance problems

**Real-world motivation:** Imagine you are an analyst who has built a quantitative stock screen. The screen flags a stock as a "buy." But how confident should you be? The screen catches 80% of winners, but it also gives false positives on 15% of losers. Bayes' theorem tells you the true probability of success -- and the answer may surprise you.

**Prerequisites:** Basic algebra.

**References:**
- CFA Institute, *CFA Program Curriculum*, Quantitative Methods.
- DeFusco, R. et al., *Quantitative Investment Analysis*, CFA Institute, Wiley.

### Why this notebook matters

Probability is the language of uncertainty, and uncertainty is the essence of finance. Every investment decision is a probability calculation: *What's the chance this stock outperforms? What's the probability of default? What's the likelihood of recession?* Without rigorous probability tools, financial decisions devolve into guesswork.

> **Key Concept:** Bayes' theorem is the most important formula in modern probabilistic reasoning. It tells you how to *update* beliefs in light of new evidence — the foundation of everything from spam filters to hedge fund alpha generation. The CFA curriculum places it front and centre because it forces analysts to think probabilistically, not deterministically.

### Outline

1. Probability rules and verification via simulation
2. Conditional probability and independence
3. Bayes' theorem — derivation and applications
4. Sequential Bayesian updating
5. Expected value, variance, and portfolio mathematics
6. Counting methods — permutations and combinations
7. References and CFA exam preparation notes

This notebook combines theoretical exposition with hands-on simulation. Every probability formula is verified by Monte Carlo experiment, providing both mathematical rigour and empirical confirmation.

> **Key Concept:** The combination of theory and simulation is intentional. Real-world financial problems often require both: theoretical understanding to set up the problem correctly, and Monte Carlo simulation to compute answers when closed-form solutions don't exist. Mastering both modes is essential for quantitative finance work.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# ── Tolerances
ATOL = 1e-10
RTOL = 1e-6

# ── Plot Style
PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

---
## 1. Probability Fundamentals

### What Is Probability?

At its core, probability assigns a number between 0 and 1 to events, measuring how likely they are to occur. A probability of 0 means "impossible" and 1 means "certain."

In finance, there are three main interpretations of probability:
- **Frequentist:** The long-run relative frequency of an event. "If we observe 10,000 similar bonds, about 200 will default" (2% default rate).
- **Subjective:** A personal degree of belief. "I believe there's a 30% chance the Fed raises rates." This is common in investment analysis.
- **A priori:** Based on logical analysis. "A fair coin has a 50% chance of heads."

### Sample Space and Events

- **Sample space** $\Omega$: the set of all possible outcomes (e.g., all possible stock returns tomorrow)
- **Event** $A \subseteq \Omega$: a subset of outcomes we care about (e.g., "the stock goes up")

### Axioms of Probability (Kolmogorov)

These three axioms are the foundation of all probability theory:

1. $0 \leq P(A) \leq 1$ for any event $A$
2. $P(\Omega) = 1$ (something must happen)
3. For mutually exclusive events $A_1, A_2, \ldots$: $P(A_1 \cup A_2 \cup \cdots) = \sum P(A_i)$

Everything else in probability theory is derived from these three axioms.

### Addition Rule: "Or" Probabilities

What is the probability that event $A$ **or** event $B$ (or both) occurs?

$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

We subtract $P(A \cap B)$ to avoid double-counting the overlap.

If $A$ and $B$ are **mutually exclusive** (they can't both happen): $P(A \cup B) = P(A) + P(B)$.

**Worked Example:** In a credit portfolio, P(Bond A defaults) = 5%, P(Bond B defaults) = 8%, P(both default) = 2%.

$$P(\text{at least one defaults}) = 0.05 + 0.08 - 0.02 = 0.11 = 11\%$$

### Multiplication Rule: "And" Probabilities

$$P(A \cap B) = P(A|B) \cdot P(B) = P(B|A) \cdot P(A)$$

### Complement Rule: "Not" Probabilities

$$P(A') = 1 - P(A)$$

In the example above: $P(\text{neither defaults}) = 1 - 0.11 = 0.89 = 89\%$.

> **CFA Exam Tip:** The addition rule is one of the most commonly tested formulas. Always check whether events are mutually exclusive before simplifying. If they are, the joint probability $P(A \cap B) = 0$.

Let's implement these rules and verify with simulation.

### The probability axioms

All of probability theory rests on three simple axioms (Kolmogorov 1933):

1. **Non-negativity:** $P(A) \geq 0$ for any event $A$
2. **Normalisation:** $P(\Omega) = 1$ — something must happen
3. **Additivity:** For mutually exclusive events $A$ and $B$, $P(A \cup B) = P(A) + P(B)$

From these three axioms, every other probability rule can be derived. This is the elegance of the Kolmogorov framework: minimal assumptions, maximal generality.

### The fundamental rules

Three rules emerge as direct consequences:

**Complement rule:**
$$P(A^c) = 1 - P(A)$$

**Addition rule (general form):**
$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

The subtraction term avoids double-counting events that are in *both* $A$ and $B$.

**Multiplication rule:**
$$P(A \cap B) = P(A) \cdot P(B|A)$$

For *independent* events, $P(B|A) = P(B)$, so the rule simplifies to $P(A \cap B) = P(A) \cdot P(B)$.

> **CFA Exam Tip:** The CFA exam tests these rules directly. A common scenario: a portfolio has a 70% chance of returning at least 8%, a 60% chance of returning at least 10%, and a 50% chance of doing both. The probability of returning at least 8% *or* at least 10% is $0.70 + 0.60 - 0.50 = 0.80$.

In [ ]:
# ── Probability rules: Implementation and verification via simulation

def addition_rule(p_a, p_b, p_a_and_b):
    """P(A ∪ B) = P(A) + P(B) - P(A ∩ B)"""
    return p_a + p_b - p_a_and_b


def complement_rule(p_a):
    """P(A') = 1 - P(A)"""
    return 1 - p_a


# ── Example: Bond default probabilities
# P(bond A defaults) = 0.05, P(bond B defaults) = 0.08
# P(both default) = 0.02 (positive default correlation)
p_a, p_b, p_ab = 0.05, 0.08, 0.02

p_a_or_b = addition_rule(p_a, p_b, p_ab)
p_neither = 1 - p_a_or_b

print("Bond Default Probabilities:")
print(f"  P(A defaults)        = {p_a:.4f}")
print(f"  P(B defaults)        = {p_b:.4f}")
print(f"  P(both default)      = {p_ab:.4f}")
print(f"  P(at least one)      = {p_a_or_b:.4f}")
print(f"  P(neither defaults)  = {p_neither:.4f}")

# ── Verify via simulation
n_sim = 1_000_000
# Direct simulation from joint table:
# P(A∩B) = 0.02, P(A∩B') = 0.03, P(A'∩B) = 0.06, P(A'∩B') = 0.89
joint_probs = np.array([0.02, 0.03, 0.06, 0.89])
cum_probs = np.cumsum(joint_probs)
categories = np.digitize(rng.random(n_sim), cum_probs)

sim_p_a = np.mean((categories == 0) | (categories == 1))
sim_p_b = np.mean((categories == 0) | (categories == 2))
sim_p_ab = np.mean(categories == 0)
sim_p_a_or_b = np.mean(categories != 3)

print(f"\nSimulation verification (n={n_sim:,}):")
print(f"  P(A)          : theory={p_a:.4f}, sim={sim_p_a:.4f}")
print(f"  P(B)          : theory={p_b:.4f}, sim={sim_p_b:.4f}")
print(f"  P(A∩B)        : theory={p_ab:.4f}, sim={sim_p_ab:.4f}")
print(f"  P(A∪B)        : theory={p_a_or_b:.4f}, sim={sim_p_a_or_b:.4f}")

**Interpreting the output:** The simulation with 1 million trials closely matches the theoretical values, confirming our formulas work correctly. The slight differences are just random sampling noise -- they would disappear with even more simulations.

Notice the key financial insight: even though each bond has a relatively low default probability (5% and 8%), the probability that **at least one** defaults is 11%. This is higher than either individual probability because there are *two* things that can go wrong. In a portfolio of many bonds, diversification doesn't eliminate default risk -- it reduces it but never to zero.**Interpreting the output:** The simulation with 1 million trials closely matches the theoretical results, validating the probability rules.

### The Law of Large Numbers in action

The convergence of simulated frequencies to theoretical probabilities is an example of the **Strong Law of Large Numbers**:

$$\lim_{n \to \infty} \frac{1}{n} \sum_{i=1}^{n} X_i = E[X] \quad \text{(almost surely)}$$

In plain English: as the number of trials grows, the sample average converges to the expected value. This is *the* foundational result that makes empirical estimation possible.

But the convergence is *slow*: the standard error shrinks only as $1/\sqrt{n}$, not $1/n$. So:
* 100 trials → standard error around 5% of the true probability
* 10,000 trials → 0.5% standard error
* 1,000,000 trials → 0.05% standard error

This $\sqrt{n}$ scaling explains why Monte Carlo simulations need millions of iterations for high-precision estimates.

> **Common Mistake:** Beginners sometimes expect their simulation to *exactly* match the theoretical answer. It will not — random noise persists at any finite sample size. The standard error tells you the magnitude of expected deviation. A simulated probability of 0.498 vs theoretical 0.500 is a perfect match if the standard error is 0.005.

---
## 2. Conditional Probability

### The Core Question: "What changes when I learn something new?"

Conditional probability answers the question: *"Given that I know B has happened, what is the probability of A?"*

This is fundamentally important in finance. For example:
- Given that we're in a recession, what is the probability this bond defaults?
- Given that a company's earnings exceeded estimates, what is the probability the stock goes up?
- Given that the Fed raised rates, what is the probability of another rate hike?

### The Formula

$$P(A|B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0$$

**Intuition:** We're "zooming in" on the world where $B$ has already happened. Out of all the scenarios where $B$ occurs, what fraction also has $A$?

**Worked Example:** From the joint probability table below:

|  | Recession (R) | Expansion (E) | Total |
|---|---|---|---|
| **Default (D)** | 0.06 | 0.02 | 0.08 |
| **No Default (D')** | 0.19 | 0.73 | 0.92 |
| **Total** | 0.25 | 0.75 | 1.00 |

$$P(D|R) = \frac{P(D \cap R)}{P(R)} = \frac{0.06}{0.25} = 0.24 = 24\%$$

$$P(D|E) = \frac{P(D \cap E)}{P(E)} = \frac{0.02}{0.75} = 0.027 = 2.7\%$$

The default probability jumps from 2.7% in an expansion to **24%** in a recession -- nearly 9 times higher! This is why the economic cycle matters so much for credit analysis.

### Independence

Events $A$ and $B$ are **independent** if and only if:
$$P(A|B) = P(A) \quad \Longleftrightarrow \quad P(A \cap B) = P(A) \cdot P(B)$$

**Intuition:** Learning that $B$ occurred gives you *no* new information about $A$.

In the example above: $P(D) \cdot P(R) = 0.08 \times 0.25 = 0.02$, but $P(D \cap R) = 0.06$. These are not equal, so default and recession are **not independent** -- they are positively associated.

> **Key Concept:** Independence means knowing one event tells you nothing about the other. In finance, most events are *not* independent -- that's what makes risk management so challenging. Defaults tend to cluster during recessions. Stock prices tend to fall together during crises.

> **Common Mistake:** Assuming independence when events are actually correlated. The 2008 financial crisis occurred partly because risk models assumed mortgage defaults were nearly independent -- they weren't.

Let's verify these calculations with code.

### Independence vs mutual exclusivity

Two concepts that students often confuse:

**Mutually exclusive events** ($A$ and $B$ cannot both occur):
$$P(A \cap B) = 0$$
$$P(A | B) = 0 \text{ (if } P(B) > 0\text{)}$$

**Independent events** (one doesn't affect the other):
$$P(A \cap B) = P(A) P(B)$$
$$P(A | B) = P(A)$$

These are *opposite* extremes. Mutually exclusive events have *maximum* dependence (knowing one rules out the other); independent events have *zero* dependence.

| | Mutually Exclusive | Independent |
|---|---|---|
| $P(A \cap B)$ | 0 | $P(A) P(B)$ |
| $P(A \cup B)$ | $P(A) + P(B)$ | $P(A) + P(B) - P(A) P(B)$ |
| $P(A \| B)$ | 0 | $P(A)$ |
| Example | "Up vs down" | "Coin flip vs dice roll" |

Most real-world events are *neither* perfectly mutually exclusive nor perfectly independent — they have intermediate degrees of dependence captured by joint probabilities.

> **CFA Exam Tip:** The exam tests this distinction directly. Phrases like "two stocks default" or "two products fail" rarely involve independence — they share systematic risk factors. Be wary of any problem that *assumes* independence without justification.

In [ ]:
def conditional_prob(p_a_and_b, p_b):
    """P(A|B) = P(A ∩ B) / P(B)"""
    assert p_b > 0, "P(B) must be positive"
    return p_a_and_b / p_b


def check_independence(p_a, p_b, p_a_and_b):
    """Check if A and B are independent: P(A∩B) = P(A)·P(B)?"""
    return np.isclose(p_a_and_b, p_a * p_b)


# ── Example: Default given economic state
joint_table = {
    'D_and_R': 0.06,  'D_and_E': 0.02,
    'ND_and_R': 0.19, 'ND_and_E': 0.73,
}

p_D = joint_table['D_and_R'] + joint_table['D_and_E']   # 0.08
p_R = joint_table['D_and_R'] + joint_table['ND_and_R']  # 0.25
p_E = joint_table['D_and_E'] + joint_table['ND_and_E']  # 0.75

# Conditional probabilities
p_D_given_R = conditional_prob(joint_table['D_and_R'], p_R)
p_D_given_E = conditional_prob(joint_table['D_and_E'], p_E)

print("Joint Probability Table: Default vs Economic State\n")
print(f"{'':>12} {'Recession':>12} {'Expansion':>12} {'Total':>8}")
print("-" * 46)
print(f"{'Default':>12} {joint_table['D_and_R']:>12.4f} {joint_table['D_and_E']:>12.4f} {p_D:>8.4f}")
print(f"{'No Default':>12} {joint_table['ND_and_R']:>12.4f} {joint_table['ND_and_E']:>12.4f} {1-p_D:>8.4f}")
print(f"{'Total':>12} {p_R:>12.4f} {p_E:>12.4f} {1.0:>8.4f}")

print(f"\nConditional Probabilities:")
print(f"  P(Default | Recession)  = {p_D_given_R:.4f}")
print(f"  P(Default | Expansion)  = {p_D_given_E:.4f}")
print(f"  P(Default) [unconditional] = {p_D:.4f}")
print(f"\n  Are Default and Recession independent? {check_independence(p_D, p_R, joint_table['D_and_R'])}")
print(f"  P(D)·P(R) = {p_D * p_R:.4f} vs P(D∩R) = {joint_table['D_and_R']:.4f}")

**Interpreting the output:**

The unconditional default probability is 8%. But conditional on a recession, it jumps to 24%. Conditional on an expansion, it drops to just 2.7%. This dramatic difference shows that the economic state provides very useful information about default risk.

The independence test confirms that $P(D) \times P(R) = 0.02 \neq P(D \cap R) = 0.06$. Default and recession are not independent -- they are positively associated (recessions cause more defaults).**Interpreting the output:**

### Conditional probability — the foundation

Conditional probability $P(A|B)$ — read "probability of $A$ given $B$" — is defined as:

$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

This is the probability of $A$ *given that* $B$ has occurred. The denominator $P(B)$ "renormalises" the probability — we restrict attention to the world where $B$ is true.

### Independence and dependence

Two events are **independent** if knowing one tells us nothing about the other:

$$P(A|B) = P(A) \quad \text{(independence)}$$

Equivalently, $P(A \cap B) = P(A) P(B)$ for independent events. Independence is a *strong* assumption — many seemingly-independent events in finance are actually correlated:
* Stock returns across companies (market beta)
* Default events across borrowers (economic cycles)
* Currency movements (correlated central bank policy)

### Why conditional probability is so important

Most real-world probability problems are conditional:
* "What's the probability of default *given* the company missed last quarter's earnings?"
* "What's the probability of recession *given* the yield curve inversion?"
* "What's the probability of fraud *given* the auditor's red-flag findings?"

In each case, we have new information (the conditioning event) and want to update our probability assessment. This is exactly what Bayes' theorem formalises — covered next.

> **Key Concept:** The conditional probability $P(A|B)$ is *generally not equal to* $P(B|A)$. Confusing the two is one of the most common errors in probabilistic reasoning. The medical test example illustrates this: $P(\text{disease}|\text{positive test}) \neq P(\text{positive test}|\text{disease})$.

---
## 3. Bayes' Theorem

### The Big Idea: Updating Beliefs with Evidence

Bayes' theorem is one of the most powerful ideas in all of statistics. It answers the question: *"Given that I just observed new evidence, how should I update my beliefs?"*

**The analogy:** Imagine you're a detective. You start with some initial suspects (your "prior" beliefs). Then new evidence arrives -- say, fingerprints at the scene. Bayes' theorem tells you exactly how to update your suspicion of each suspect based on the new evidence. The more consistent the evidence is with a particular suspect, the more your suspicion shifts toward them.

### Derivation

From the definition of conditional probability:
$$P(A|B) = \frac{P(A \cap B)}{P(B)} = \frac{P(B|A) \cdot P(A)}{P(B)}$$

Using the **law of total probability** for the denominator:
$$P(B) = \sum_{i=1}^{n} P(B|A_i) \cdot P(A_i)$$

where $\{A_1, \ldots, A_n\}$ partition the sample space.

### Bayes' Theorem (Full Form)

$$\boxed{P(A_i|B) = \frac{P(B|A_i) \cdot P(A_i)}{\sum_{j=1}^{n} P(B|A_j) \cdot P(A_j)}}$$

**Terminology:**
- $P(A_i)$: **Prior** probability (your belief before seeing the evidence)
- $P(B|A_i)$: **Likelihood** (how likely the evidence is under each hypothesis)
- $P(A_i|B)$: **Posterior** probability (your updated belief after seeing the evidence)
- $P(B)$: **Evidence** (total probability of seeing the data)

The formula can be remembered as:

$$\text{Posterior} = \frac{\text{Likelihood} \times \text{Prior}}{\text{Evidence}}$$

### The Classic Example: Adapted for Finance

A disease screening test is the classic way to build Bayes' intuition. Let's use it first, then apply the same logic to finance.

**Setup:** A test for a rare condition has:
- Prevalence: P(Disease) = 1% (the "base rate")
- Sensitivity: P(Test+ | Disease) = 95% (catches 95% of sick people)
- False positive rate: P(Test+ | No Disease) = 5%

**Question:** If you test positive, what is the probability you actually have the disease?

**Most people guess 90-95%.** The correct answer is shockingly low -- let's work through it step by step.

**Step 1:** Imagine 10,000 people taking the test.
- 100 have the disease (1% of 10,000)
- 9,900 don't have it

**Step 2:** Of the 100 with disease, 95 test positive (95% sensitivity).

**Step 3:** Of the 9,900 without disease, 495 test positive (5% false positive rate).

**Step 4:** Total positive tests: 95 + 495 = 590.

**Step 5:** P(Disease | Test+) = 95 / 590 = **16.1%**

> **Key Concept:** When the base rate is low, even a very accurate test produces mostly false positives. This same principle applies to stock screens, trading signals, and risk models. A "95% accurate" trading signal might still be wrong most of the time if profitable trades are rare.

Let's implement Bayes' theorem and visualize it as a probability tree.

### Bayes' theorem — the formula

Combining the definitions of conditional probability:

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

Each component has a name:
* **Prior** $P(A)$: belief about $A$ *before* observing evidence
* **Likelihood** $P(B|A)$: probability of evidence *given* $A$ is true
* **Marginal** $P(B)$: total probability of evidence (normalising constant)
* **Posterior** $P(A|B)$: updated belief about $A$ *after* observing evidence

$$\text{Posterior} = \frac{\text{Likelihood} \times \text{Prior}}{\text{Marginal}}$$

### Why Bayes' theorem is profound

Bayes' theorem describes *how rational beliefs update*. Given:
1. A prior belief about a hypothesis
2. New evidence
3. The likelihood of that evidence under each hypothesis

The theorem produces the *uniquely correct* updated belief. Any other update procedure violates probability axioms.

This is why Bayes' theorem appears everywhere:
* **Medical diagnosis:** Combining test results with disease prevalence
* **Spam filtering:** Combining word frequencies with prior spam rates
* **Hedge fund alpha:** Updating views on stocks based on new information
* **Credit risk:** Updating default probabilities as data arrives
* **Machine learning:** Bayesian inference and probabilistic models

> **CFA Exam Tip:** The CFA exam expects mastery of Bayes' theorem. Practice the canonical structure: identify the prior, identify the likelihood, identify the marginal, apply the formula. Most exam problems can be solved by carefully labelling these components.

In [ ]:
def bayes_theorem(priors, likelihoods):
    """Compute posterior probabilities using Bayes' theorem.
    
    Parameters
    ----------
    priors      : array-like -- P(A_i) for each hypothesis
    likelihoods : array-like -- P(B|A_i) for each hypothesis
    
    Returns
    -------
    posteriors  : ndarray -- P(A_i|B) for each hypothesis
    evidence    : float   -- P(B) = total probability of evidence
    """
    priors = np.asarray(priors, dtype=float)
    likelihoods = np.asarray(likelihoods, dtype=float)
    
    assert np.isclose(np.sum(priors), 1.0), "Priors must sum to 1"
    
    # Joint probabilities: P(B|A_i) * P(A_i)
    joint = likelihoods * priors
    
    # Evidence: P(B) = sum of joints
    evidence = np.sum(joint)
    
    # Posteriors: P(A_i|B) = joint_i / evidence
    posteriors = joint / evidence
    
    return posteriors, evidence


# ── Classic screening test example
priors = [0.01, 0.99]           # [P(Disease), P(No Disease)]
likelihoods = [0.95, 0.05]      # [P(+|Disease), P(+|No Disease)]

posteriors, p_positive = bayes_theorem(priors, likelihoods)

print("Classic Bayes Example: Screening Test\n")
print(f"  Prior P(Disease)       = {priors[0]:.4f}")
print(f"  P(Test+|Disease)       = {likelihoods[0]:.4f}")
print(f"  P(Test+|No Disease)    = {likelihoods[1]:.4f}")
print(f"\n  P(Test+) [evidence]    = {p_positive:.4f}")
print(f"  P(Disease|Test+)       = {posteriors[0]:.4f}")
print(f"  P(No Disease|Test+)    = {posteriors[1]:.4f}")
print(f"\n  Insight: Despite a positive test, there is only a {posteriors[0]:.1%} chance of disease!")
print(f"  The low base rate (1%) overwhelms the test's accuracy.")

**Why is this result so counterintuitive?** The key is the base rate. With only 1% prevalence, the "pool" of healthy people is 99 times larger than the pool of sick people. Even though the test is wrong on only 5% of healthy people, 5% of a very large number still produces more false positives than true positives.

This is directly relevant to finance: if only 30% of stocks outperform the benchmark, a stock screen that's "80% accurate" will still have a significant false positive rate. We'll see this exact application shortly.

The following probability tree makes the calculation visually clear.**Why is this result so counterintuitive?** The key is the base rate. With only 1% prevalence, even a high-quality test produces many more false positives than true positives.

### The base rate fallacy — the most important cognitive bias in finance

The base rate fallacy occurs when people focus on the test accuracy (likelihood) and ignore the prior probability (base rate). Bayes' theorem rigorously combines both.

### A worked example

A test for a rare disease (1% prevalence) has 95% sensitivity and 95% specificity. Someone tests positive. What's the probability they have the disease?

Most people guess 95%. The correct answer:

$$P(\text{disease} | +) = \frac{0.95 \times 0.01}{0.95 \times 0.01 + 0.05 \times 0.99} \approx 16\%$$

Even after a positive test, there's only a 16% chance the person has the disease! Why? Because:
* Of 1000 people tested, 10 have the disease — 9.5 of them test positive
* But 990 don't have it, and 49.5 of them *also* test positive (false positives)
* Total positives: 59 — only 9.5 are true positives
* $9.5 / 59 \approx 16\%$

### Applications in finance

The base rate fallacy is everywhere:

* **Stock screens:** A "value stock signal" with 70% historical accuracy is mostly false positives if only 5% of stocks actually outperform.

* **Credit scoring:** A "default warning indicator" with 90% sensitivity catches most actual defaults — but most companies that trip the indicator never default.

* **Fraud detection:** Even a sophisticated algorithm has many false positives because most companies are legitimate.

> **Common Mistake:** Investors and analysts routinely commit the base rate fallacy. They evaluate signals by their *accuracy* (likelihood) without considering the *base rate*. A "signal" with 80% accuracy may produce mostly false positives if the underlying base rate is 5%.

In [ ]:
# ── Visualization: Bayes' theorem as a tree diagram
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-1.5, 1.5)
ax.axis('off')
ax.set_title("Bayes' Theorem: Probability Tree", fontsize=14, fontweight='bold')

# Tree structure
ax.annotate('', xy=(0.5, 0.7), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color=PRIMARY, lw=2))
ax.annotate('', xy=(0.5, -0.7), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color=SECONDARY, lw=2))

ax.text(0.15, 0.45, f'P(D) = {priors[0]}', fontsize=11, color=PRIMARY)
ax.text(0.05, -0.55, f"P(D') = {priors[1]}", fontsize=11, color=SECONDARY)

for y_start, p_pos, p_neg, color, label in [
    (0.7, likelihoods[0], 1-likelihoods[0], PRIMARY, 'D'),
    (-0.7, likelihoods[1], 1-likelihoods[1], SECONDARY, "D'")]:
    
    ax.annotate('', xy=(1.8, y_start + 0.3), xytext=(0.6, y_start),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    ax.annotate('', xy=(1.8, y_start - 0.3), xytext=(0.6, y_start),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5, linestyle='--'))
    ax.text(0.9, y_start + 0.35, f'P(+|{label})={p_pos}', fontsize=10)
    ax.text(0.9, y_start - 0.2, f'P(-|{label})={p_neg}', fontsize=10)

joints = [
    (1.0, f'P(D∩+) = {priors[0]*likelihoods[0]:.4f}'),
    (0.4, f'P(D∩-) = {priors[0]*(1-likelihoods[0]):.4f}'),
    (-0.4, f"P(D'∩+) = {priors[1]*likelihoods[1]:.4f}"),
    (-1.0, f"P(D'∩-) = {priors[1]*(1-likelihoods[1]):.4f}"),
]
for y, text in joints:
    ax.text(2.0, y, text, fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

ax.text(0.5, -1.3, f"Posterior: P(D|+) = P(D∩+)/P(+) = {priors[0]*likelihoods[0]:.4f}/{p_positive:.4f} = {posteriors[0]:.4f}",
        fontsize=12, fontweight='bold', color='darkred',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='mistyrose', alpha=0.8))

plt.tight_layout()
plt.show()

**Reading the probability tree:**

Follow the branches from left to right:
1. The first branch splits by disease status (1% vs 99%).
2. The second branch splits by test result (positive/negative) conditional on disease status.
3. At the leaves, we get the joint probabilities by multiplying along the path.
4. The posterior is computed by dividing the joint probability of interest by the total probability of the evidence.

The key visual insight: the "Disease and Test+" leaf (0.0095) is **much smaller** than the "No Disease and Test+" leaf (0.0495). This is why most positive tests are false positives.**Reading the probability tree:**

### The probability tree as a calculation tool

Probability trees are pedagogically powerful because they make every Bayes' calculation explicit:

* **Branches** represent possible outcomes
* **Branch probabilities** are the conditional probabilities at each node
* **Path probabilities** are products along a complete path
* **Sums of paths** give marginal probabilities

To compute a Bayesian posterior:
1. Build the tree from priors and likelihoods
2. Identify the paths that match the evidence
3. Sum probabilities of those paths to get the marginal
4. Take the ratio of "favourable paths" to "total paths"

This visual approach prevents algebraic errors and builds intuition.

### When trees become unwieldy

Trees work well for simple problems (2-3 hypotheses, 1-2 pieces of evidence). For complex problems:
* Many hypotheses → many branches
* Sequential evidence → exponentially growing tree
* Continuous distributions → trees become impossibly complex

For these cases, formal Bayesian inference (often using probabilistic programming) replaces tree-based reasoning. But the underlying logic is identical.

> **Key Concept:** The probability tree is more than a teaching tool — it is a *complete representation* of any discrete probabilistic problem. If you can correctly draw the tree, you can solve any Bayesian inference problem mechanically. The skill being tested is *modelling*, not just computation.

---
## 4. Bayes in Finance

Now let's apply Bayes' theorem to three practical financial scenarios. The mathematics is identical to the screening test -- only the context changes.

### Application 1: Stock Screening

You build a quantitative screen that flags stocks as "buy" candidates.
- **Base rate:** Only 30% of stocks actually outperform the benchmark
- **Sensitivity:** The screen correctly flags 80% of the stocks that will outperform
- **False positive rate:** The screen also flags 15% of the stocks that will underperform

**Question:** If your screen flags a stock, what is the probability it actually outperforms?

### Application 2: Updating Economic State Beliefs

Your prior belief: 20% chance of recession. Then a leading economic indicator turns negative.
- P(Negative indicator | Recession) = 70%
- P(Negative indicator | Expansion) = 10%

**Question:** After seeing the negative indicator, what is the updated recession probability?

### Application 3: Manager Skill Assessment

Only 10% of fund managers have true skill (generate alpha). A manager has outperformed 3 years in a row.
- P(Outperform 3 years | Skilled) = 75%
- P(Outperform 3 years | Unskilled) = 12.5% (just luck -- 0.5^3)

**Question:** After 3 years of outperformance, what is the probability the manager is truly skilled?

> **CFA Exam Tip:** Bayes' theorem problems on the exam always follow the same pattern: you're given priors and likelihoods, and asked to compute a posterior. Set up the formula methodically -- identify the prior, the likelihood, and compute the evidence.

Let's compute all three.

### Sequential Bayesian updating

When new evidence arrives over time, the posterior from one round becomes the prior for the next round. This is **sequential updating**:

$$\text{Initial} \to \text{Posterior}_1 \to \text{Posterior}_2 \to \text{Posterior}_3 \to \ldots$$

Each step refines the belief. Asymptotically, the posterior converges to the truth (for well-specified models with sufficient data).

### Updating in finance

Bayesian updating is the formal foundation of:

* **Black-Litterman portfolio optimisation:** Combining market priors with investor views
* **Credit risk monitoring:** Updating default probabilities as financial reports arrive
* **Algorithmic trading:** Updating predictions in real-time as price data streams in
* **Macro forecasting:** Updating recession probabilities as economic data releases

> **CFA Exam Tip:** Sequential updating is conceptually simple but easily mishandled. The key insight: today's posterior is tomorrow's prior. As long as each piece of evidence is conditionally independent given the hypothesis, you can chain updates indefinitely.

### Bayesian thinking in finance — beyond formulas

Bayes' theorem is more than a formula — it is a *worldview*. Bayesian thinking has three core principles:

1. **Beliefs are quantitative:** Express uncertainty as probabilities, not as "high/medium/low" labels.

2. **Update with evidence:** When new data arrives, update beliefs according to Bayes' rule. Don't ignore evidence; don't over-react to it.

3. **Multiple hypotheses:** Hold multiple competing hypotheses simultaneously, with probability mass distributed across them.

The Bayesian financial analyst:
* Doesn't say "the stock will go up" — says "the probability of outperformance is 65%"
* Doesn't ignore conflicting evidence — incorporates it into the posterior
* Doesn't claim certainty — acknowledges the residual uncertainty even after analysis

> **Key Concept:** Bayesian reasoning forces intellectual honesty. By expressing beliefs as probabilities, you commit to specific quantitative claims. Other people can challenge those claims with new evidence, and your beliefs must update accordingly. This is why Bayesian thinking is the gold standard for rational decision-making under uncertainty.

In [ ]:
# ── Application 1: Stock screening test
priors_stock = [0.30, 0.70]
likelihoods_stock = [0.80, 0.15]

posteriors_stock, p_screen_pos = bayes_theorem(priors_stock, likelihoods_stock)

print("Stock Screening Analysis:")
print(f"  Prior P(Outperform)          = {priors_stock[0]:.2f}")
print(f"  P(Screen+|Outperform)        = {likelihoods_stock[0]:.2f}")
print(f"  P(Screen+|Underperform)      = {likelihoods_stock[1]:.2f}")
print(f"  P(Screen+)                   = {p_screen_pos:.4f}")
print(f"  P(Outperform|Screen+)        = {posteriors_stock[0]:.4f}")
print(f"\n  The screen raises confidence from {priors_stock[0]:.0%} to {posteriors_stock[0]:.1%}")

# ── Application 2: Updating recession probability
priors_econ = [0.20, 0.80]
likelihoods_econ = [0.70, 0.10]

posteriors_econ, p_neg = bayes_theorem(priors_econ, likelihoods_econ)

print(f"\nEconomic State Update (after negative leading indicator):")
print(f"  Prior P(Recession)          = {priors_econ[0]:.2f}")
print(f"  Posterior P(Recession|Neg)   = {posteriors_econ[0]:.4f}")
print(f"  Recession probability jumped from {priors_econ[0]:.0%} to {posteriors_econ[0]:.1%}")

# ── Application 3: Manager skill
priors_mgr = [0.10, 0.90]
likelihoods_mgr = [0.75, 0.125]

posteriors_mgr, p_outperf = bayes_theorem(priors_mgr, likelihoods_mgr)

print(f"\nManager Skill Assessment (after 3 years outperformance):")
print(f"  Prior P(Skilled)             = {priors_mgr[0]:.2f}")
print(f"  Posterior P(Skilled|3yr out)  = {posteriors_mgr[0]:.4f}")
print(f"  Still only a {posteriors_mgr[0]:.1%} chance the manager is truly skilled!")

**Interpreting the three applications:**

1. **Stock screening:** The screen raises your confidence from 30% to about 70%. That's useful, but it still means nearly 1 in 3 flagged stocks will underperform. A naive investor might think an "80% accurate" screen means 80% of its picks will be winners -- Bayes' theorem shows the truth is more nuanced.

2. **Economic update:** A single negative indicator more than doubles the recession probability from 20% to about 64%. This shows how a highly diagnostic signal (one that's 7x more likely in a recession than an expansion) can dramatically shift beliefs.

3. **Manager skill:** Even after 3 years of outperformance, there's still only about a 40% chance the manager is truly skilled! This is because the prior (only 10% of managers have skill) is so low. The lesson: **track records are weaker evidence of skill than most investors assume.**

> **Common Mistake:** Ignoring the base rate. A 3-year track record of outperformance might feel convincing, but when 90% of managers are unskilled, even luck can produce impressive short-term results.

### Sequential Updating: How Many Years of Outperformance Does It Take?

What if the manager keeps beating the benchmark year after year? Let's track how the posterior evolves over time.**Interpreting the three applications:**

### Why these applications matter

The three applications demonstrate Bayes' theorem in distinct financial contexts:

1. **Stock screening:** Updating beliefs about stock outperformance based on signals (technical, fundamental, or quantitative)

2. **Credit assessment:** Updating default probabilities based on financial indicators (debt ratios, profitability trends, cash flow patterns)

3. **Forecasting accuracy:** Updating beliefs about analyst skill based on track records

In each case, the same logical structure applies: prior + likelihood + evidence → posterior. Mastering this pattern lets you analyse a wide range of financial problems systematically.

### The screening test framework

A common pattern: a "test" classifies entities as positive or negative for some condition (outperformance, default, fraud). The Bayes' framework requires:

* **Prior** $P(\text{condition})$: base rate
* **Sensitivity** $P(+|\text{condition})$: true positive rate
* **Specificity** $P(-|\text{not condition})$: true negative rate
* **Posterior** $P(\text{condition}|+)$: updated probability after positive test

The key insight from these applications: **even good tests have high false positive rates when the base rate is low**. This is why investors should be sceptical of any single screening result and demand multiple confirming signals.

> **Common Mistake:** Practitioners often quote the "accuracy" of a model (e.g., 80% accurate) without considering whether this matters at the relevant base rate. A model with 80% accuracy on a 50/50 base rate is useful; the same accuracy on a 5/95 base rate may produce mostly false positives.

In [ ]:
# ── Visualization: Sequential Bayesian updating
p_skilled = 0.10  # prior
p_beat_skilled = 0.65    # P(beat | skilled)
p_beat_unskilled = 0.50  # P(beat | unskilled) = coin flip

years = 15
posterior_history = [p_skilled]

for year in range(years):
    posteriors, _ = bayes_theorem(
        [p_skilled, 1 - p_skilled],
        [p_beat_skilled, p_beat_unskilled]
    )
    p_skilled = posteriors[0]
    posterior_history.append(p_skilled)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(years + 1), posterior_history, 'o-', color=PRIMARY, linewidth=2, markersize=8)
ax.axhline(0.5, color=SECONDARY, linestyle='--', alpha=0.7, label='50% threshold')
ax.set_xlabel('Years of Consecutive Outperformance')
ax.set_ylabel('P(Manager is Skilled)')
ax.set_title('Sequential Bayesian Update: Manager Skill Assessment')
ax.set_ylim(0, 1)
ax.legend()

for i in [0, 5, 10, years]:
    ax.annotate(f'{posterior_history[i]:.1%}', (i, posterior_history[i]),
                textcoords='offset points', xytext=(0, 15), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

**Interpreting the chart:**

Starting from just 10% confidence in the manager's skill, each year of outperformance nudges the posterior upward. It takes about **6-7 years** of consecutive outperformance just to reach 50% confidence! And even after 15 years of beating the benchmark every single year, we're still not at 100% -- there's always some residual probability of prolonged luck.

This has profound implications for the investment industry:
- Short track records are nearly meaningless for distinguishing skill from luck
- The prior (base rate of skilled managers) matters enormously
- This explains why even Nobel laureate-run funds (like LTCM) can blow up -- past performance truly does not guarantee future results

> **CFA Exam Tip:** Bayes' theorem on the CFA exam is always about updating probabilities with new information. The pattern: Prior + Evidence = Posterior. Practice setting up the formula with the correct prior and likelihood.**Interpreting the chart:**

### Reading the sequential update plot

The chart shows how the posterior probability evolves as additional evidence arrives. Key features:

1. **Starting point:** The prior probability — initial belief before any evidence
2. **First update:** Posterior shifts based on first piece of evidence
3. **Subsequent updates:** Each additional evidence narrows or refines the belief
4. **Convergence:** With enough informative evidence, the posterior converges toward the truth

### What determines the rate of convergence?

Two factors govern how fast Bayesian beliefs converge:

1. **Quality of evidence:** Higher likelihood ratios update faster. A test with sensitivity 99% / specificity 99% updates beliefs much faster than one with 60% / 60%.

2. **Strength of prior:** A *strong* prior (high confidence) is harder to overturn. A *weak* prior (low confidence) shifts more dramatically with new evidence.

This creates a delicate balance:
* Too strong a prior → slow to learn from evidence
* Too weak a prior → over-react to noisy evidence

> **Key Concept:** The choice of prior is one of the most controversial aspects of Bayesian analysis. Critics argue that subjective priors taint analysis with personal beliefs. Bayesians respond that *all* analysis involves implicit priors — making them explicit (via Bayes' theorem) is more honest than hiding them in frequentist methods.

---
## 5. Expected Value & Variance

### Expected Value: The Probability-Weighted Average

The expected value of a random variable is the "center of gravity" of its distribution -- the long-run average if you repeated the experiment many times.

$$E[X] = \sum_{i} x_i \cdot P(x_i) \quad \text{(discrete)}$$

**Investment meaning:** The expected return of a stock is the probability-weighted average of all possible returns. It's your best single-number forecast of what the return will be.

**Worked Example:** A stock has these possible returns next year:

| Scenario | Probability | Return |
|---|---|---|
| Boom | 25% | +30% |
| Normal | 50% | +10% |
| Recession | 25% | -15% |

$$E[R] = 0.25 \times 0.30 + 0.50 \times 0.10 + 0.25 \times (-0.15) = 0.075 + 0.05 - 0.0375 = 8.75\%$$

### Variance: Quantifying Risk

Variance measures the spread of possible outcomes around the expected value:

$$\text{Var}(X) = E[(X - \mu)^2] = \sum_i P(x_i)(x_i - \mu)^2$$

The standard deviation $\sigma = \sqrt{\text{Var}(X)}$ is in the same units as the returns, making it easier to interpret.

**Investment meaning:** Variance (or standard deviation) is the most common measure of **total risk** for a single asset. Higher variance means more uncertainty about the actual return.

### Covariance and Correlation: How Assets Move Together

$$\text{Cov}(X, Y) = E[(X - \mu_X)(Y - \mu_Y)]$$

$$\rho_{XY} = \frac{\text{Cov}(X,Y)}{\sigma_X \sigma_Y}, \quad -1 \leq \rho \leq 1$$

**Investment meaning:** Covariance and correlation measure how two assets move *together*. Negative correlation means they tend to move in opposite directions -- exactly what we want for diversification!

### Portfolio Expected Return & Variance (2 assets)

$$E[R_p] = w_1 E[R_1] + w_2 E[R_2]$$
$$\sigma_p^2 = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2 w_1 w_2 \text{Cov}(R_1, R_2)$$

> **Key Concept:** Portfolio return is always the weighted average of individual returns. But portfolio *risk* can be **less** than the weighted average of individual risks -- that's the benefit of diversification, and it works because of the covariance term.

> **CFA Exam Tip:** The portfolio variance formula is the most important formula in portfolio theory. Make sure you can compute it quickly. The covariance term is what makes diversification work -- if $\text{Cov} < 0$, it *reduces* portfolio risk.

Let's compute everything for a stock-bond portfolio under three economic scenarios.

### Expected value — the foundation of decision theory

The expected value $E[X]$ is the *probability-weighted average* of all possible outcomes:

$$E[X] = \sum_{i} x_i \cdot P(X = x_i)$$

For continuous distributions, the sum becomes an integral. The expected value is the long-run average if you could repeat the random experiment infinitely many times.

### Why expected value matters

In finance, expected value drives:

* **Investment decisions:** Choose investments with higher expected return
* **Insurance pricing:** Premium = expected loss + risk loading
* **Option pricing:** BSM is fundamentally an expected value calculation
* **Project evaluation:** Net present value uses expected cash flows

### The expected value isn't the most likely outcome

A common confusion: $E[X]$ is *not* the most likely outcome. It is the long-run average. For asymmetric distributions (lognormal, lottery payoffs), the median (most likely) and mean (expected value) can differ dramatically.

**Example:** A lottery has 99% chance of paying \$0 and 1% chance of paying \$1 million. Expected value = \$10,000. Most likely outcome = \$0. The expected value tells you the long-run average over many lotteries; it doesn't tell you what happens on any single play.

> **CFA Exam Tip:** When evaluating investment options, the CFA curriculum tests both expected value and *variance* (or standard deviation). Two options with the same expected value but different variances are not equivalent — the higher-variance option is *riskier*. Mean-variance analysis (covered in Portfolio Theory) formalises this trade-off.

### The St. Petersburg paradox — when expected value misleads

In 1738, Daniel Bernoulli posed a thought experiment: a coin is flipped until it lands tails. If the first tails occurs on flip $n$, the payoff is $2^n$ dollars.

Expected value:
$$E[X] = \sum_{n=1}^{\infty} \frac{1}{2^n} \cdot 2^n = \sum_{n=1}^{\infty} 1 = \infty$$

The expected payoff is *infinite*. Yet most people would not pay even \$10 to play this game. Why?

The resolution: people don't maximize expected *dollar* value — they maximize expected *utility*. The marginal utility of money is *decreasing* (each additional dollar adds less satisfaction than the previous one). With logarithmic utility, the expected utility of the St. Petersburg game is finite, matching observed willingness-to-pay.

This insight launched modern utility theory and ultimately led to the development of risk aversion as a foundational concept in finance.

> **CFA Exam Tip:** The CFA curriculum doesn't test the St. Petersburg paradox directly, but the underlying concept — that humans don't maximize expected dollar value — is central to risk aversion, utility-based portfolio choice, and behavioral finance. Understanding the paradox helps you grasp why expected value alone is insufficient for evaluating investments.

In [ ]:
def expected_value(outcomes, probabilities):
    """E[X] = sum(x_i * p_i)"""
    outcomes = np.asarray(outcomes, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)
    assert np.isclose(np.sum(probabilities), 1.0)
    return np.sum(outcomes * probabilities)


def variance_discrete(outcomes, probabilities):
    """Var(X) = E[X^2] - (E[X])^2"""
    outcomes = np.asarray(outcomes, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)
    mu = expected_value(outcomes, probabilities)
    return np.sum(probabilities * (outcomes - mu) ** 2)


def covariance_discrete(x_outcomes, y_outcomes, joint_probs):
    """Compute Cov(X,Y) from joint distribution."""
    x = np.asarray(x_outcomes, dtype=float)
    y = np.asarray(y_outcomes, dtype=float)
    jp = np.asarray(joint_probs, dtype=float)
    p_x = np.sum(jp, axis=1)
    p_y = np.sum(jp, axis=0)
    mu_x = np.sum(x * p_x)
    mu_y = np.sum(y * p_y)
    e_xy = 0
    for i in range(len(x)):
        for j in range(len(y)):
            e_xy += x[i] * y[j] * jp[i, j]
    return e_xy - mu_x * mu_y


def portfolio_return_variance(w, mu, cov_matrix):
    """Portfolio expected return and variance."""
    w = np.asarray(w, dtype=float)
    mu = np.asarray(mu, dtype=float)
    cov = np.asarray(cov_matrix, dtype=float)
    port_return = w @ mu
    port_var = w @ cov @ w
    return port_return, port_var


# ── Example: Stock return under different economic scenarios
scenarios = ['Boom', 'Normal', 'Recession']
probs = [0.25, 0.50, 0.25]
stock_returns = [0.30, 0.10, -0.15]
bond_returns = [0.02, 0.06, 0.12]

e_stock = expected_value(stock_returns, probs)
e_bond = expected_value(bond_returns, probs)
var_stock = variance_discrete(stock_returns, probs)
var_bond = variance_discrete(bond_returns, probs)

cov_sb = sum(p * (rs - e_stock) * (rb - e_bond) 
             for p, rs, rb in zip(probs, stock_returns, bond_returns))
corr_sb = cov_sb / (np.sqrt(var_stock) * np.sqrt(var_bond))

print(f"{'Scenario':<12} {'Prob':>6} {'Stock':>8} {'Bond':>8}")
print("-" * 36)
for s, p, rs, rb in zip(scenarios, probs, stock_returns, bond_returns):
    print(f"{s:<12} {p:>6.2f} {rs:>8.2%} {rb:>8.2%}")

print(f"\n{'E[R]':<12} {'':>6} {e_stock:>8.2%} {e_bond:>8.2%}")
print(f"{'σ':<12} {'':>6} {np.sqrt(var_stock):>8.2%} {np.sqrt(var_bond):>8.2%}")
print(f"\nCov(Stock, Bond) = {cov_sb:.6f}")
print(f"Corr(Stock, Bond) = {corr_sb:.4f}")
print(f"\nNegative correlation: stocks and bonds diversify each other!")

# ── 60/40 portfolio
w = np.array([0.60, 0.40])
mu_vec = np.array([e_stock, e_bond])
cov_mat = np.array([[var_stock, cov_sb], [cov_sb, var_bond]])

port_ret, port_var = portfolio_return_variance(w, mu_vec, cov_mat)
print(f"\n60/40 Portfolio:")
print(f"  E[R_p] = {port_ret:.2%}")
print(f"  σ_p    = {np.sqrt(port_var):.2%}")

**Interpreting the results:**

Notice the key numbers:
- **Stocks:** Expected return 8.75% with 15.8% standard deviation (high risk, high reward)
- **Bonds:** Expected return 6.50% with 3.5% standard deviation (lower risk, lower reward)
- **Correlation: -0.99** -- stocks and bonds move in almost perfectly opposite directions in this scenario setup (bonds rally during recessions when stocks fall)

The **60/40 portfolio** delivers 7.85% expected return with just 8.8% standard deviation. Compare this to holding 100% stocks (8.75% return, 15.8% risk). By giving up less than 1% of expected return, you cut risk nearly in half! That's the power of diversification with negatively correlated assets.

Let's visualize the risk-return tradeoff for all possible stock/bond allocations.**Interpreting the results:**

### Variance and risk in expected value calculations

Expected value alone is insufficient — variance (or standard deviation) measures risk:

$$\text{Var}(X) = E[(X - E[X])^2] = E[X^2] - (E[X])^2$$

For a discrete distribution:

$$\text{Var}(X) = \sum_i (x_i - E[X])^2 \cdot P(X = x_i)$$

Standard deviation is the square root of variance, in the same units as $X$.

### Properties of variance

Three useful properties:

1. **$\text{Var}(aX + b) = a^2 \text{Var}(X)$** — adding a constant doesn't affect variance; multiplying scales variance by the square.

2. **For independent variables:** $\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y)$ — variances add.

3. **For correlated variables:** $\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y) + 2\text{Cov}(X, Y)$ — covariance must be added.

### Diversification — the magic of correlation

The third property explains *diversification*. If two assets have variances of 16% each and are *uncorrelated*, a 50/50 portfolio has variance:

$$\text{Var} = 0.5^2 \times 0.16 + 0.5^2 \times 0.16 = 0.08$$

The portfolio variance is *lower* than either component's variance, even though both have the same individual risk. This "free lunch" is why diversification reduces risk without sacrificing return — the foundation of modern portfolio theory.

> **Key Concept:** Variance is the bridge between probability theory and modern finance. It quantifies risk in a way that combines linearly across portfolios, enabling tractable optimisation. Markowitz's mean-variance framework — which won the 1990 Nobel Prize — is built directly on the variance properties above.

In [ ]:
# ── Visualization: Portfolio risk-return for different weights
weights_stock = np.linspace(0, 1, 100)
port_rets = []
port_stds = []

for ws in weights_stock:
    w_vec = np.array([ws, 1 - ws])
    ret, var = portfolio_return_variance(w_vec, mu_vec, cov_mat)
    port_rets.append(ret)
    port_stds.append(np.sqrt(var))

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(np.array(port_stds) * 100, np.array(port_rets) * 100, 
                c=weights_stock, cmap='coolwarm', s=20)
plt.colorbar(sc, label='Weight in Stocks')

ax.plot(np.sqrt(var_stock) * 100, e_stock * 100, 'o', color=PRIMARY, markersize=12, 
        label='100% Stocks', zorder=5)
ax.plot(np.sqrt(var_bond) * 100, e_bond * 100, 's', color=SECONDARY, markersize=12, 
        label='100% Bonds', zorder=5)
ax.plot(np.sqrt(port_var) * 100, port_ret * 100, '*', color=TERTIARY, markersize=15, 
        label='60/40 Portfolio', zorder=5)

ax.set_xlabel('Risk (Standard Deviation, %)')
ax.set_ylabel('Expected Return (%)')
ax.set_title('Portfolio Efficient Frontier: Stocks vs Bonds')
ax.legend()
plt.tight_layout()
plt.show()

**Reading the efficient frontier:**

The curve shows every possible combination of stocks and bonds. Notice that it bows to the **left** -- this means some portfolios have less risk than *either* individual asset! This is the visual proof of diversification.

The minimum-risk portfolio is somewhere around 20-25% stocks, where the curve reaches its leftmost point. Even adding a small amount of stocks to a bond portfolio can actually *reduce* total risk because of the negative correlation.**Reading the efficient frontier:**

### The efficient frontier — modern portfolio theory in one chart

The efficient frontier shows the set of *efficient* portfolios — those that achieve the highest expected return for each level of risk. Key features:

1. **The curve slopes upward:** Higher returns require higher risk. There is no free lunch.

2. **The shape is concave (curves backward):** This reflects diminishing returns to risk-taking. The marginal return per unit of additional risk decreases at the high-risk end.

3. **The minimum-variance portfolio** sits at the leftmost point. It is the portfolio with the lowest possible risk given the available assets.

4. **The capital market line** (not shown but conceptually relevant) is a tangent line from the risk-free rate to the efficient frontier — its tangency point is the *market portfolio*.

### Implications for investors

Modern portfolio theory makes a startling claim: every rational investor should hold a combination of:
* The risk-free asset
* The market portfolio (the tangency portfolio)

The *only* difference between conservative and aggressive investors is the proportion in each. Both face the same investment opportunity set; they just sit at different points on the capital market line.

While this idealised result doesn't hold perfectly in reality (investors have heterogeneous beliefs, transaction costs, and constraints), it provides the conceptual framework for index investing and asset allocation.

> **CFA Exam Tip:** The efficient frontier is a foundational concept tested extensively in Portfolio Management. At Level 1, focus on understanding what it represents and how diversification expands the achievable risk-return space. At Level 2, the math of constructing the frontier becomes important.

---
## 6. Counting Methods

### Why Counting Matters

Counting methods let you figure out how many ways something can happen, which is essential for computing probabilities. In finance, they show up in portfolio construction (how many portfolios can you build from 20 stocks?) and in risk calculations.

### Factorial

$$n! = n \cdot (n-1) \cdot \ldots \cdot 1, \quad 0! = 1$$

### Permutations (order matters)

"How many ways can you *arrange* $k$ items from $n$?"

$$P(n, k) = \frac{n!}{(n-k)!}$$

**Example:** How many ways can you rank the top 3 stocks from 10 candidates? $P(10,3) = 10 \times 9 \times 8 = 720$.

### Combinations (order doesn't matter)

"How many ways can you *choose* $k$ items from $n$?"

$$C(n, k) = \binom{n}{k} = \frac{n!}{k!(n-k)!}$$

**Example:** How many different 5-stock portfolios can you build from 20 candidates? $C(20,5) = 15{,}504$.

> **CFA Exam Tip:** Use permutations when the order of selection matters (ranking analysts, sequencing investments). Use combinations when order doesn't matter (selecting stocks for a portfolio).

Let's implement these from scratch and apply them to finance.

### Why counting matters

Many probability calculations reduce to *counting*:
* "How many ways can event $A$ occur?"
* "How many total outcomes are possible?"
* "Probability = count favourable / count total"

For complex events, counting requires careful application of permutations and combinations. The CFA curriculum tests these counting tools because they appear repeatedly:
* Computing portfolio return distributions
* Analysing tournament-style investment competitions
* Calculating arrangement probabilities in trading sequences
* Combinatorial optimization in active management

### The four basic counting principles

1. **Multiplication principle:** If event $A$ has $n_A$ outcomes and event $B$ has $n_B$ outcomes, then the joint event has $n_A \times n_B$ outcomes.

2. **Addition principle:** If events are mutually exclusive, total outcomes = sum of individual counts.

3. **Permutations:** Number of ordered arrangements of $r$ items from $n$ is $P(n, r) = \frac{n!}{(n-r)!}$

4. **Combinations:** Number of unordered selections of $r$ items from $n$ is $C(n, r) = \binom{n}{r} = \frac{n!}{r!(n-r)!}$

Permutations and combinations differ by the factor $r!$ — the number of orderings of $r$ items. When order matters, multiply by $r!$; when order doesn't matter, divide it out.

> **Common Mistake:** A canonical exam trap: "How many ways can a portfolio manager pick 5 stocks from 20?" If the order matters (they're assigned to different positions), the answer is $P(20, 5) = 1,860,480$. If the stocks are interchangeable, the answer is $C(20, 5) = 15,504$. Always read the problem carefully to determine whether order matters.

In [ ]:
def factorial(n):
    """Compute n! from scratch."""
    if n <= 1:
        return 1
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


def permutations(n, k):
    """P(n,k) = n! / (n-k)!"""
    return factorial(n) // factorial(n - k)


def combinations(n, k):
    """C(n,k) = n! / (k! * (n-k)!)"""
    return factorial(n) // (factorial(k) * factorial(n - k))


def multinomial(n, groups):
    """Multinomial coefficient: n! / (n1! * n2! * ... * nk!)"""
    assert sum(groups) == n, "Groups must sum to n"
    denom = 1
    for g in groups:
        denom *= factorial(g)
    return factorial(n) // denom


# ── Portfolio selection: choosing 5 stocks from 20 candidates
n_universe = 20
n_select = 5
n_portfolios = combinations(n_universe, n_select)
print(f"Choosing {n_select} stocks from {n_universe}: C({n_universe},{n_select}) = {n_portfolios:,} possible portfolios")

# Ranking: how many ways to rank top 3 from 10 stocks?
n_rank = permutations(10, 3)
print(f"Ranking top 3 from 10: P(10,3) = {n_rank:,} orderings")

# Multinomial: allocate 12 analysts into 3 teams of 4
n_allocations = multinomial(12, [4, 4, 4])
print(f"Allocating 12 analysts into 3 teams of 4: {n_allocations:,} ways")

# ── Application: Probability of specific portfolio composition
# From 20 stocks (8 tech, 7 healthcare, 5 finance), 
# pick 5 at random. P(exactly 2 tech, 2 healthcare, 1 finance)?
favorable = combinations(8, 2) * combinations(7, 2) * combinations(5, 1)
total = combinations(20, 5)
prob = favorable / total
print(f"\nP(2 tech, 2 healthcare, 1 finance out of 5) = {favorable}/{total} = {prob:.4f}")

**Interpreting the results:**

There are 15,504 different 5-stock portfolios you can build from just 20 stocks. This number grows astronomically with a larger universe -- for the S&P 500 with 500 stocks, there are over $2.6 \times 10^{12}$ possible 5-stock portfolios. This is why systematic portfolio construction methods are essential.

The probability calculation shows that if you randomly pick 5 stocks from a universe of 20 (8 tech, 7 healthcare, 5 finance), there's about a 19% chance of getting exactly 2 tech, 2 healthcare, and 1 finance. This is a hypergeometric probability -- useful for understanding random sampling from stratified populations.

---
## References

1. **CFA Institute**, *CFA Program Curriculum Level I*, Quantitative Methods: Probability Concepts.
2. **DeFusco, R., McLeavey, D., Pinto, J., & Runkle, D.**, *Quantitative Investment Analysis*, 3rd ed., CFA Institute/Wiley, 2015.
3. **Ross, S.**, *A First Course in Probability*, 10th ed., Pearson, 2019.
4. **Bertsekas, D. & Tsitsiklis, J.**, *Introduction to Probability*, 2nd ed., Athena Scientific, 2008.**Interpreting the results:**

### Why permutations and combinations matter

Counting techniques underpin many financial calculations:

* **Combinations** count *unordered* arrangements: how many distinct portfolios of 5 stocks can be selected from 100? Answer: $\binom{100}{5}$.

* **Permutations** count *ordered* arrangements: how many distinct trading sequences of 5 trades can be made from 100 candidate trades? Answer: $\binom{100}{5} \times 5! = P(100, 5)$.

The choice depends on whether order matters.

### Applications in finance

* **Option strategies:** A "spread" combines two or more options. The number of possible spreads from $n$ options grows combinatorially.

* **Portfolio construction:** Selecting $k$ stocks from $n$ candidates produces $\binom{n}{k}$ portfolios — a vast space even for modest $n$.

* **Permutation tests:** Statistical method to test hypotheses by computing the test statistic under all possible permutations of the data.

* **Combinatorial optimization:** Some portfolio problems (cardinality-constrained optimization) require enumerating combinations.

### Computational considerations

Combinatorial counts grow extremely fast:

| n | $\binom{n}{n/2}$ | $n!$ |
|---|------------------|------|
| 10 | 252 | 3.6M |
| 20 | 184,756 | 2.4 quintillion |
| 50 | $1.3 \times 10^{14}$ | $3.0 \times 10^{64}$ |

For large $n$, exhaustive enumeration is infeasible — algorithms must use smart search techniques (dynamic programming, branch-and-bound, heuristics).

> **Key Concept:** Combinatorial counting is the foundation of *enumerative probability* — when outcomes can be enumerated, probabilities are ratios of counts. For events with infinite or unmanageable outcome spaces, we must turn to integration and continuous probability — the subject of distribution theory.

> **Common Mistake:** Counting problems often look "easy" but contain subtle traps. Always carefully identify: (1) Is order important? (2) Are repetitions allowed? (3) Are items distinct? Misanswering any of these leads to the wrong formula and wrong answer.

> **Final note:** Mastery of the concepts in this notebook is essential for the CFA Level 1 exam, as well as for practical financial analysis work. Practice the worked examples by hand and verify your understanding by reproducing the code from scratch.
